# Fine-tune mT5-base — Tóm Tắt + Đề Xuất Tag (Multi-task)

**Yêu cầu Kaggle:**
- Accelerator: **GPU T4 x2** hoặc **P100**
- Internet: **ON** (lần đầu tải model)
- Upload `vnexpress_dataset.json` vào phần **Input** (Add Data > Upload)

**Chiến lược tăng chất lượng:**
- Input = `[title] + [content]` → nhiều ngữ cảnh hơn
- mT5-base (~580M params) thay vì small (~300M)
- AdaFactor optimizer (thiết kế cho T5)
- Label smoothing để tránh overfit
- Early stopping theo ROUGE-L

**Ước tính:** ~4–6 giờ với T4 x2, ~10,000 bài

In [ ]:
!pip install transformers datasets sentencepiece accelerate rouge-score -q

In [ ]:
import json, random, os
import numpy as np
import torch
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    MT5ForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
)
from rouge_score import rouge_scorer

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}  : {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory // 1024**3} GB)')

In [ ]:
# ════════════════════════════════════════════════════════════════
# CẤU HÌNH
# ════════════════════════════════════════════════════════════════

# mT5-base: ~580MB, tốt hơn small đáng kể với cùng data
MODEL_NAME = 'google/mt5-base'

MAX_INPUT_LENGTH  = 512   # tokens input (title + content)
MAX_TARGET_LENGTH = 128   # tokens output
BATCH_SIZE        = 4     # mỗi GPU (base lớn hơn → giảm batch)
GRAD_ACCUM        = 4     # effective batch = 4 * 2GPU * 4 = 32
EPOCHS            = 6
LEARNING_RATE     = 3e-4  # phù hợp AdaFactor
WARMUP_RATIO      = 0.08
LABEL_SMOOTHING   = 0.1   # tránh overfit
SEED              = 42

# Đường dẫn dataset — đổi nếu khác
DATA_PATH  = '/kaggle/input/vnexpress-dataset/vnexpress_dataset.json'
OUTPUT_DIR = Path('/kaggle/working/mt5-multitask')
FINAL_DIR  = Path('/kaggle/working/mt5-multitask-final')
OUTPUT_DIR.mkdir(exist_ok=True)
FINAL_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# ════════════════════════════════════════════════════════════════
# LOAD & KIỂM TRA DATA
# ════════════════════════════════════════════════════════════════
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    articles = json.load(f)

print(f'Tổng bài viết: {len(articles)}')

from collections import Counter
src_count   = Counter(a.get('source', 'unknown') for a in articles)
word_counts = [len(a['content'].split()) for a in articles]
tag_counts  = [len(a['tags']) for a in articles]

print(f'Nguồn  : {dict(src_count)}')
print(f'Content: min={min(word_counts)} | avg={int(np.mean(word_counts))} | max={max(word_counts)} từ')
print(f'Tags   : avg={np.mean(tag_counts):.1f} | max={max(tag_counts)}')

# Xem mẫu
s = articles[0]
print(f'\n--- Mẫu ---')
print(f'Title  : {s["title"]}')
print(f'Summary: {s["summary"][:120]}')
print(f'Tags   : {s["tags"]}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# XÂY DỰNG TRAINING EXAMPLES
# ════════════════════════════════════════════════════════════════
# Input format: "<task>: <title> </s> <content>"
# Dùng </s> ngăn cách title và content giống cách T5 xử lý multi-segment

def build_examples(articles):
    examples = []
    for item in articles:
        title   = item['title'].strip()
        summary = item['summary'].strip()
        tags    = item['tags']
        # Giữ tối đa 350 từ content để dành chỗ cho title trong 512 tokens
        content = ' '.join(item['content'].split()[:350])

        inp_base = f"{title} </s> {content}"

        # ── Task 1: Tóm tắt ──────────────────────────────────────────────────
        if len(summary) >= 40:
            examples.append({
                'input_text' : f'summarize: {inp_base}',
                'target_text': summary,
                'task'       : 'summarize',
            })

        # ── Task 2: Đề xuất tag ───────────────────────────────────────────────
        if len(tags) >= 2:
            # Normalize tag: lowercase, strip
            clean_tags = [t.strip().lower() for t in tags[:5]]
            examples.append({
                'input_text' : f'suggest tags: {inp_base}',
                'target_text': ', '.join(clean_tags),
                'task'       : 'suggest tags',
            })

    return examples

all_examples = build_examples(articles)
random.shuffle(all_examples)

split = int(len(all_examples) * 0.9)
train_examples = all_examples[:split]
val_examples   = all_examples[split:]

n_sum  = sum(1 for e in all_examples if e['task'] == 'summarize')
n_tags = sum(1 for e in all_examples if e['task'] == 'suggest tags')

print(f'Tổng examples : {len(all_examples):,}')
print(f'  Tóm tắt      : {n_sum:,}')
print(f'  Đề xuất tag  : {n_tags:,}')
print(f'  Train        : {len(train_examples):,}')
print(f'  Val          : {len(val_examples):,}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# LOAD MODEL & TOKENIZER
# ════════════════════════════════════════════════════════════════
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = MT5ForConditionalGeneration.from_pretrained(MODEL_NAME)

n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params/1e6:.0f}M')

# Kiểm tra token </s> có trong vocab không
print(f'EOS token: "{tokenizer.eos_token}" (id={tokenizer.eos_token_id})')

In [ ]:
# ════════════════════════════════════════════════════════════════
# TOKENIZE
# ════════════════════════════════════════════════════════════════
def tokenize_fn(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_list(train_examples)
val_ds   = Dataset.from_list(val_examples)

train_tok = train_ds.map(
    tokenize_fn, batched=True, batch_size=256,
    remove_columns=['input_text', 'target_text', 'task'],
    desc='Tokenize train'
)
val_tok = val_ds.map(
    tokenize_fn, batched=True, batch_size=256,
    remove_columns=['input_text', 'target_text', 'task'],
    desc='Tokenize val'
)

print(f'Train: {len(train_tok):,} | Val: {len(val_tok):,}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# METRICS (ROUGE)
# ════════════════════════════════════════════════════════════════
scorer_obj = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    r1, r2, rL = [], [], []
    for p, l in zip(decoded_preds, decoded_labels):
        sc = scorer_obj.score(l.strip(), p.strip())
        r1.append(sc['rouge1'].fmeasure)
        r2.append(sc['rouge2'].fmeasure)
        rL.append(sc['rougeL'].fmeasure)

    return {
        'rouge1': round(np.mean(r1) * 100, 2),
        'rouge2': round(np.mean(r2) * 100, 2),
        'rougeL': round(np.mean(rL) * 100, 2),
    }

In [ ]:
# ════════════════════════════════════════════════════════════════
# TRAINING ARGUMENTS
# ════════════════════════════════════════════════════════════════
training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,

    # AdaFactor: optimizer thiết kế riêng cho T5, tiết kiệm memory hơn Adam
    optim='adafactor',
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type='cosine',

    # Chống overfit
    label_smoothing_factor=LABEL_SMOOTHING,

    # Eval & checkpoint
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rougeL',
    greater_is_better=True,
    save_total_limit=2,

    # Generation
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,

    # Tối ưu memory
    fp16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    group_by_length=True,  # nhóm examples cùng độ dài → giảm padding → nhanh hơn

    logging_steps=100,
    report_to='none',
    seed=SEED,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

effective_batch = BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
print('Config tóm tắt:')
print(f'  Model          : {MODEL_NAME}')
print(f'  Train examples : {len(train_tok):,}')
print(f'  Effective batch: {effective_batch}')
print(f'  Epochs         : {EPOCHS} (có early stopping)')
print(f'  Optimizer      : AdaFactor (lr={LEARNING_RATE})')
print(f'  Label smoothing: {LABEL_SMOOTHING}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# TRAIN
# ════════════════════════════════════════════════════════════════
print('Bắt đầu training...')
train_result = trainer.train()
print('Training hoàn tất!')
print(f'  Train loss cuối: {train_result.training_loss:.4f}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# LƯU MODEL
# ════════════════════════════════════════════════════════════════
trainer.save_model(str(FINAL_DIR))
tokenizer.save_pretrained(str(FINAL_DIR))
print(f'Model saved: {FINAL_DIR}')

In [ ]:
# ════════════════════════════════════════════════════════════════
# TEST THỬ KẾT QUẢ
# ════════════════════════════════════════════════════════════════
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Nếu dùng DataParallel thì lấy module gốc
raw_model = model.module if hasattr(model, 'module') else model
raw_model.to(device)

def predict(title, content, task='summarize', max_new_tokens=128):
    inp = f"{task}: {title} </s> {' '.join(content.split()[:350])}"
    inputs = tokenizer(inp, return_tensors='pt',
                       max_length=MAX_INPUT_LENGTH, truncation=True).to(device)
    with torch.no_grad():
        out = raw_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            length_penalty=1.2,       # khuyến khích output dài hơn
            no_repeat_ngram_size=3,
            early_stopping=True,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

# Test 5 bài cuối (model chưa thấy)
for i, art in enumerate(articles[-5:]):
    print(f"\n{'='*60}")
    print(f"BÀI {i+1}: {art['title']}")
    pred_sum  = predict(art['title'], art['content'], 'summarize')
    pred_tags = predict(art['title'], art['content'], 'suggest tags', max_new_tokens=64)
    print(f"[TÓM TẮT THỰC]    {art['summary'][:150]}")
    print(f"[TÓM TẮT DỰ ĐOÁN] {pred_sum}")
    print(f"[TAGS THỰC]        {art['tags']}")
    print(f"[TAGS DỰ ĐOÁN]     {pred_tags}")

In [ ]:
# ════════════════════════════════════════════════════════════════
# ĐÁNH GIÁ ROUGE TÁCH BIỆT TỪNG TASK
# ════════════════════════════════════════════════════════════════
from tqdm import tqdm

def eval_task(examples, task_name, n=150):
    subset = [e for e in examples if e['task'] == task_name][:n]
    r1, r2, rL = [], [], []
    for ex in tqdm(subset, desc=f'Eval [{task_name}]'):
        # Tái tạo title/content từ input_text
        raw = ex['input_text'].replace(f'{task_name}: ', '', 1)
        parts = raw.split(' </s> ', 1)
        title   = parts[0] if len(parts) == 2 else ''
        content = parts[1] if len(parts) == 2 else raw
        pred = predict(title, content, task_name)
        ref  = ex['target_text']
        sc = scorer_obj.score(ref, pred)
        r1.append(sc['rouge1'].fmeasure)
        r2.append(sc['rouge2'].fmeasure)
        rL.append(sc['rougeL'].fmeasure)
    print(f'  ROUGE-1={np.mean(r1)*100:.2f}  ROUGE-2={np.mean(r2)*100:.2f}  ROUGE-L={np.mean(rL)*100:.2f}')

print('\n=== Kết quả đánh giá trên tập Validation ===')
eval_task(val_examples, 'summarize')
eval_task(val_examples, 'suggest tags')

In [ ]:
# ════════════════════════════════════════════════════════════════
# NÉN VÀ DOWNLOAD
# ════════════════════════════════════════════════════════════════
import shutil
zip_path = '/kaggle/working/mt5-multitask-final'
shutil.make_archive(zip_path, 'zip', str(FINAL_DIR))

size_mb = os.path.getsize(f'{zip_path}.zip') / 1024**2
print(f'File zip: {zip_path}.zip  ({size_mb:.0f} MB)')
print('Vào tab Output của Kaggle để download về máy.')
print('Sau đó giải nén vào thư mục final_model_mt5/ trong project.')